# B3Z — Zoomed SAM2 (small-object scale)

## Why
Dense-scene error is **75–80% merging**, and every SAM2 change so far kept the image scale fixed (multimask, re-prompting seeds, masked input, SAM2.1-Large, negative points). At 384 px, packed objects are only a few pixels wide in SAM2's 1024-px input. **Cropping magnifies objects 2×** — a change of scale, not capacity.

## Pre-declared design (before any B3Z result)
| Choice | Rule |
|---|---|
| B0-Z | automatic mask generator, `points_per_side` 32, **`crop_n_layers` 1** (full image + 2 × 2 overlapping crops, each resized to SAM2's input; SAM2's own crop merging); otherwise default, `apply_postprocessing=False` |
| Guided-Z | crop boxes from SAM2's `generate_crop_boxes` (layer 1, same overlap as the generator); each cached `dense_recovery` prompt is predicted in the layer-1 crop where it lies **furthest from the crop border**, mask placed back into full-image coordinates; one positive point, multimask off |
| Bases | **[S]** SAM2.1-Small, full image; **[LM0]** SAM2.1-Large, masked input M0 (target-winning tiled region; masks > 50% outside discarded), as B3I / B3L §5 |
| Counting | unchanged: GPV_R4 (NMS 0.5 + B0 pool), then H1b |
| Compared with | H1b_v1 and H1b-L[M0] (unzoomed bases) |
| Targets | T1, T2′, T3′ |

## Run order
T4 → 1.0–1.2 → 2.1–2.2 → set `RUN_BASES` in 3.1 → 3.1 (resumable) → 4.1.
To run the two bases in parallel: copy this notebook, set `RUN_BASES = ["S"]` in one and `["LM0"]` in the other.

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
OUT_DIR = RESULTS_ROOT / "B3Z_Zoomed_SAM2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.hybrid as hybrid
import src.pipeline.b0 as b0
for m in (io_cache, sg, gpv, hybrid, b0):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
print("Repo commit:", REPO_COMMIT, "| output:", OUT_DIR)

Repo commit: d70bb1d6d631c12156ba7bf1881d22f5b2eb6100 | output: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3Z_Zoomed_SAM2


In [3]:
import torch

# ============================================================
# Step 1.2 — Shared helpers: guided masks, B0 meta filtering, systems, targets
# ============================================================

def guided_masks_for_prompts(predictor, image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

def filter_meta(meta, keep):
    return {k: np.asarray(v)[keep] for k, v in meta.items()}

def systems_from_masks(b0_masks, b0_iou, g_masks, g_scores, a3c, depth, san_target):
    keep = gpv.nms_masks(g_masks, g_scores, 0.5) if g_masks.shape[0] else np.zeros(0, int)
    r4_m, _, _ = gpv.build_r4(g_masks[keep], np.asarray(g_scores)[keep], b0_masks, b0_iou, 0.5)
    d1 = gpv.d1_peaks(san_target)
    h1b = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    return {"B0": int(b0_masks.shape[0]), "GPV_R4": int(r4_m.shape[0]), "H1b": h1b}

def evaluate(frame, systems, label):
    gt = frame["gt_count"].to_numpy(float); st = frame["density_stratum"].to_numpy()
    AE = {s: np.abs(frame[s].to_numpy(float) - gt) for s in systems}
    IDX = {}
    for j, k in enumerate(STRATA):
        pos = np.flatnonzero(st == k)
        IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]
    def sci(a, b, k):
        d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
        return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
    def rci(a, b):
        d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
        o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
        return o, np.percentile(d, 2.5), np.percentile(d, 97.5)
    res = pd.DataFrame([{"system": s,
                         "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                         "mean_signed": float((frame[s] - frame["gt_count"]).mean()),
                         **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in systems])
    rows = []
    for s in [x for x in systems if x not in ("B0_v1", "GPV_R4_v1", "H1b_v1")]:
        rec = {"system": s}; t1 = []; t2 = []
        for k in ["S1_0_20", "S2_20_40"]:
            d, lo, hi = sci(s, "B0_v1", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
        for k in ["S5_80_95", "S6_95_100"]:
            d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
        d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
        rows.append(rec)
    print(f"{label} — metrics:"); display(res.round(2))
    print("Targets (difference [95% CI]; negative = system better):"); display(pd.DataFrame(rows))
    return res, pd.DataFrame(rows)
print("helpers ready")

helpers ready


# 2. SAM2 models

In [4]:
# ============================================================
# Step 2.1 — Install SAM2 (frozen commit)
# ============================================================

import torch
assert torch.cuda.is_available(), "Switch the runtime to T4."
DEVICE = "cuda"
%cd /content
if not Path("/content/sam2_repo/sam2/__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")
importlib.invalidate_caches()
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
print("SAM2 commit:", subprocess.check_output(["git", "-C", "/content/sam2_repo", "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

/content
/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.8 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
/content
SAM2 commit: 2b90b9f5ceec907a1c18123530e92e794ad901a4
GPU: Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0


In [5]:
# ============================================================
# Step 2.2 — Build zoomed AMGs and predictors (Small and Large)
# ============================================================

from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
from sam2.utils.amg import generate_crop_boxes

SAM2_S = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_small.pt"
SAM2_L = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_large.pt"
CFG_L = "configs/sam2.1/sam2.1_hiera_l.yaml"
assert b0.sha256_file(SAM2_S) == b0.FROZEN_CHECKPOINT_SHA256 and SAM2_L.exists()

model_S = build_sam2(b0.FROZEN_MODEL_CFG, str(SAM2_S), device=DEVICE, apply_postprocessing=False).eval()
model_L = build_sam2(CFG_L, str(SAM2_L), device=DEVICE, apply_postprocessing=False).eval()
amg_S_Z = SAM2AutomaticMaskGenerator(model_S, points_per_side=32, crop_n_layers=1)
amg_L_Z = SAM2AutomaticMaskGenerator(model_L, points_per_side=32, crop_n_layers=1)
predictor_S = SAM2ImagePredictor(build_sam2(b0.FROZEN_MODEL_CFG, str(SAM2_S), device=DEVICE).eval())
predictor_L = SAM2ImagePredictor(build_sam2(CFG_L, str(SAM2_L), device=DEVICE).eval())
CROP_OVERLAP = amg_S_Z.crop_overlap_ratio
print("Zoomed AMGs (crop_n_layers=1, crop_overlap_ratio =", CROP_OVERLAP, ") and predictors ready")

Zoomed AMGs (crop_n_layers=1, crop_overlap_ratio = 0.3413333333333333 ) and predictors ready


# 3. Zoomed caching

In [7]:
# ============================================================
# Step 3.1 — B0-Z and guided-Z on train_dev400 (resumable per base)
# ============================================================

RUN_BASES = ["S", "LM0"]          # set to ["S"] or ["LM0"] to run the bases in two parallel sessions
TD_TILED = TD_ROOT / "san_vocab_tiled"
Z_ROOT = TD_ROOT / "zoomed"

ZERO_MASK_IMAGES = []          # images where SAM2's crop merging hit its empty-result bug

def amg_masks(amg, image, tag=None):
    # SAM2's AMG with crop_n_layers >= 1 crashes (IndexError in box_area) when an image
    # yields no masks at all; treat that case as "no masks", as SAM2 returns without crops.
    try:
        return b0.run_amg(amg, image)
    except IndexError:
        ZERO_MASK_IMAGES.append(tag)
        h, w = image.shape[:2]
        return np.zeros((0, h, w), bool), {"areas": np.zeros(0, np.int64), "predicted_iou": np.zeros(0, np.float32),
                                          "stability_score": np.zeros(0, np.float32), "bbox_xywh": np.zeros((0, 4)),
                                          "crop_box_xywh": np.zeros((0, 4)), "point_xy": np.zeros((0, 2))}

def layer1_crops(h, w):
    boxes, layers = generate_crop_boxes((h, w), 1, CROP_OVERLAP)
    return [b for b, l in zip(boxes, layers) if l == 1]          # [x0, y0, x1, y1]

def guided_zoomed(predictor, image, prompts):
    h, w = image.shape[:2]
    if len(prompts) == 0:
        return np.zeros((0, h, w), bool), np.zeros(0)
    crops = layer1_crops(h, w)
    margin = np.array([[min(p[0] - x0, x1 - 1 - p[0], p[1] - y0, y1 - 1 - p[1]) for (x0, y0, x1, y1) in crops]
                       for p in prompts])
    choice = margin.argmax(axis=1)
    masks = np.zeros((len(prompts), h, w), bool); scores = np.zeros(len(prompts))
    with torch.inference_mode():
        for c, (x0, y0, x1, y1) in enumerate(crops):
            idx = np.flatnonzero(choice == c)
            if len(idx) == 0:
                continue
            predictor.set_image(np.ascontiguousarray(image[y0:y1, x0:x1]))
            for k in idx:
                m, s, _ = predictor.predict(point_coords=np.asarray([[prompts[k][0] - x0, prompts[k][1] - y0]], np.float32),
                                            point_labels=np.asarray([1], np.int32), multimask_output=False)
                masks[k, y0:y1, x0:x1] = np.asarray(m[0], bool); scores[k] = float(np.asarray(s)[0])
    return masks, scores

def keep_inside(masks, region):
    if masks.shape[0] == 0:
        return np.zeros(0, bool)
    inside = (masks & region[None]).reshape(masks.shape[0], -1).sum(1)
    return inside > 0.5 * np.maximum(masks.reshape(masks.shape[0], -1).sum(1), 1)

for base in RUN_BASES:
    for d in ["b0", "guided", "_done"]:
        (Z_ROOT / base / d).mkdir(parents=True, exist_ok=True)
    todo = [r for r in td_df.itertuples(index=False)
            if not (Z_ROOT / base / "_done" / f"{int(r.dataset_index):05d}.json").exists()]
    print(f"[{base}] done: {len(td_df) - len(todo)} | to process: {len(todo)}")
    t0 = time.time()
    for i, r in enumerate(tqdm(todo, desc=f"Zoomed [{base}]")):
        di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
        img = b0.load_image_rgb_cv2(FSC147_IMAGE_DIR / r.image_id)
        prompts = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")["prompts_xy"]
        if base == "S":
            bm, bmeta = amg_masks(amg_S_Z, img, ("S", int(di)))
            gm, gs = guided_zoomed(predictor_S, img, prompts)
            kb, kg = np.ones(len(bm), bool), np.ones(len(gm), bool)
        else:
            with np.load(TD_TILED / f"{stem}.npz") as v:
                reg = v["tiled_argmax"] == 0
            masked = (img * reg[..., None]).astype(np.uint8)
            bm, bmeta = amg_masks(amg_L_Z, masked, ("LM0", int(di)))
            gm, gs = guided_zoomed(predictor_L, masked, prompts)
            kb, kg = keep_inside(bm, reg), keep_inside(gm, reg)
        io_cache.save_b0_masks(Z_ROOT / base / "b0" / f"{int(di):05d}_{stem}.npz", bm[kb], filter_meta(bmeta, kb))
        io_cache.save_guided_masks(Z_ROOT / base / "guided" / f"{int(di):04d}_{stem}.npz", gm[kg],
                                   prompts[kg] if len(prompts) else prompts, gs[kg], img.shape[:2],
                                   dataset_index=di, image_id=r.image_id)
        with open(Z_ROOT / base / "_done" / f"{int(di):05d}.json", "w") as f:
            json.dump({"dataset_index": int(di), "b0": int(kb.sum()), "guided": int(kg.sum())}, f)
        if (i + 1) % 20 == 0:
            torch.cuda.empty_cache()
    print(f"[{base}] elapsed: {(time.time() - t0) / 60:.1f} min")

print("Images where zoomed AMG found no masks:", ZERO_MASK_IMAGES)

[S] done: 182 | to process: 218


Zoomed [S]:   0%|          | 0/218 [00:00<?, ?it/s]

[S] elapsed: 98.2 min
[LM0] done: 0 | to process: 400


Zoomed [LM0]:   0%|          | 0/400 [00:00<?, ?it/s]

[LM0] elapsed: 215.8 min
Images where zoomed AMG found no masks: [('S', 1561)]


# 4. Systems and targets

In [8]:
# ============================================================
# Step 4.1 — GPV_R4-Z / H1b-Z for both bases; targets (CPU-capable)
# ============================================================

Z_ROOT = TD_ROOT / "zoomed"
bases_ready = [b for b in ["S", "LM0"] if len(list((Z_ROOT / b / "_done").glob("*.json"))) == 400] \
    if Z_ROOT.exists() else []
print("Bases complete:", bases_ready)
rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Systems"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    row = {"dataset_index": di}
    for base in bases_ready:
        bm, bmeta = io_cache.load_b0_masks(Z_ROOT / base / "b0" / f"{int(di):05d}_{stem}.npz")
        g = io_cache.load_guided_masks(Z_ROOT / base / "guided" / f"{int(di):04d}_{stem}.npz")
        s = systems_from_masks(bm, bmeta["predicted_iou"], g["masks"], g["scores"], a3c, dep, san["target_score"])
        row.update({f"{k}-Z[{base}]": v for k, v in s.items()})
    rows.append(row)
zf = pd.DataFrame(rows)
lm = pd.read_csv(RESULTS_ROOT / "B3L_SAM2_Large" / "section5_large_masked" / "B3L_s5_per_image.csv")[
    ["dataset_index", "B0-L[M0]", "GPV_R4-L[M0]", "H1b-L[M0]"]]
frame = (zf.merge(lm, on="dataset_index")
         .merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
                .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one"))
print("B0 masks per image (unzoomed vs zoomed), by stratum:")
cols_b0 = ["B0_v1"] + [c for c in ["B0-Z[S]", "B0-L[M0]", "B0-Z[LM0]"] if c in frame.columns]
display(frame.groupby("density_stratum")[cols_b0].mean().reindex(STRATA).round(1))

SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1", "H1b-L[M0]"] + \
      [f"{k}-Z[{b}]" for b in bases_ready for k in ["B0", "GPV_R4", "H1b"]]
res, targets = evaluate(frame, SYS, "Zoomed SAM2 (train_dev400)")

frame.to_csv(OUT_DIR / "B3Z_per_image.csv", index=False)
res.to_csv(OUT_DIR / "B3Z_metrics.csv", index=False)
targets.to_csv(OUT_DIR / "B3Z_targets.csv", index=False)
with open(OUT_DIR / "B3Z_metadata.json", "w") as f:
    json.dump({"notebook": "B3Z_Zoomed_SAM2",
               "b0_z": "AMG points_per_side 32, crop_n_layers 1, apply_postprocessing False",
               "guided_z": "layer-1 crops from generate_crop_boxes; prompt predicted in the crop where it is "
                           "furthest from the border; mask mapped back; multimask off",
               "bases": {"S": "SAM2.1-Small, full image", "LM0": "SAM2.1-Large, masked input M0 (discard >50% outside)"},
               "bases_evaluated": bases_ready,
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Bases complete: ['S', 'LM0']


Systems:   0%|          | 0/400 [00:00<?, ?it/s]

B0 masks per image (unzoomed vs zoomed), by stratum:


,B0_v1,B0-Z[S],B0-L[M0],B0-Z[LM0]
density_stratum,,,,
S1_0_20,11.8,17.4,11.1,15.2
S2_20_40,16.4,22.1,16.4,20.8
S3_40_60,21.0,28.3,23.0,31.6
S4_60_80,33.1,48.8,33.2,45.2
S5_80_95,54.9,75.3,52.7,75.7
S6_95_100,69.2,149.2,61.4,144.0


Zoomed SAM2 (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,H1b-L[M0],28.91,-49.22,6.89,6.14,9.93,14.90,42.42,299.43
4,B0-Z[S],30.70,-41.89,8.86,9.65,14.08,21.38,46.37,258.88
5,GPV_R4-Z[S],30.01,-27.92,15.18,13.05,17.85,20.20,35.35,229.07
6,H1b-Z[S],29.73,-24.72,16.30,14.45,18.70,20.08,32.45,219.05
7,B0-Z[LM0],29.09,-43.36,6.45,8.45,11.23,18.63,44.93,267.90
8,GPV_R4-Z[LM0],27.53,-36.70,9.40,8.16,14.75,15.95,36.18,248.98
9,H1b-Z[LM0],27.28,-33.92,10.54,8.95,15.90,16.12,32.32,242.67


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,H1b-L[M0],"+1.33 [-0.44, +3.11]","-1.60 [-3.34, +0.16]","+5.08 [-0.08, +11.52]","+24.45 [+12.60, +38.02]","+0.26 [-1.03, +1.64]",True,False,False,False
1,B0-Z[S],"+3.30 [+1.61, +5.51]","+1.91 [+0.36, +3.55]","+9.03 [+2.55, +16.47]","-16.10 [-36.53, +3.22]","+2.05 [+0.16, +3.98]",False,False,False,False
2,GPV_R4-Z[S],"+9.61 [+7.21, +12.16]","+5.31 [+2.85, +7.96]","-1.98 [-7.42, +4.85]","-45.92 [-63.60, -29.48]","+1.36 [-0.40, +3.24]",False,True,False,False
3,H1b-Z[S],"+10.74 [+8.11, +13.61]","+6.71 [+3.95, +9.59]","-4.88 [-8.53, -1.03]","-55.93 [-73.43, -40.28]","+1.08 [-0.52, +2.78]",False,True,False,False
4,B0-Z[LM0],"+0.89 [-0.59, +2.51]","+0.71 [-1.02, +2.56]","+7.60 [-0.35, +16.82]","-7.08 [-25.72, +11.75]","+0.44 [-1.49, +2.42]",True,True,False,False
5,GPV_R4-Z[LM0],"+3.84 [+1.65, +6.19]","+0.42 [-1.75, +2.70]","-1.15 [-8.12, +7.25]","-26.00 [-44.10, -6.83]","-1.12 [-2.84, +0.68]",False,True,False,False
6,H1b-Z[LM0],"+4.97 [+2.62, +7.50]","+1.21 [-1.18, +3.80]","-5.02 [-10.83, +2.20]","-32.32 [-51.35, -12.13]","-1.37 [-3.03, +0.38]",False,True,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3Z_Zoomed_SAM2
